In [25]:

import os
import copy
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from torch import nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_scheduler
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# 1. 基础配置
# ============================================================

MODEL_NAME = 'google-bert/bert-base-chinese'

BATCH_SIZE = 16
MAX_LENGTH = 256
EPOCHS = 10

LEARNING_RATE = 5e-4

PATIENCE = 3

MODEL_PATH = 'best_model.pth'

LABEL_NAMES = ['负面', '正面']

SEED = 42


def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

print('当前设备:', device)


# ============================================================
# 2. 加载 Tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)


# ============================================================
# 3. 加载数据集
# ============================================================

class MyDataset(Dataset):

    def __init__(self, split):

        self.dataset = load_dataset(
            'lansinuote/ChnSentiCorp',
            split=split
        )

    def __len__(self):

        return len(self.dataset)

    def __getitem__(self, index):

        item = self.dataset[index]

        return (
            item['text'],
            item['label']
        )


train_dataset = MyDataset('train')
val_dataset = MyDataset('validation')
test_dataset = MyDataset('test')


print('训练集数量:', len(train_dataset))
print('验证集数量:', len(val_dataset))
print('测试集数量:', len(test_dataset))


# ============================================================
# 4. 数据整理函数
# ============================================================

def collate_fn(batch):

    sentences = [
        sent for sent, _ in batch
    ]

    labels = [
        label for _, label in batch
    ]

    data = tokenizer(
        sentences,
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
        return_tensors='pt'
    )

    return {
        'input_ids': data['input_ids'],
        'attention_mask': data['attention_mask'],
        'token_type_ids': data['token_type_ids'],
        'labels': torch.tensor(
            labels,
            dtype=torch.long
        )
    }


# ============================================================
# 5. DataLoader
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    drop_last=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    drop_last=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    drop_last=False
)


# ============================================================
# 6. 加载预训练 BERT
# ============================================================

pretrained_model = AutoModel.from_pretrained(
    MODEL_NAME
)


# ============================================================
# 7. 定义分类模型
# ============================================================

class Model(nn.Module):

    def __init__(self, pretrained_model):

        super().__init__()

        self.bert = pretrained_model

        hidden_size = self.bert.config.hidden_size

        self.fc = nn.Linear(
            hidden_size,
            2
        )

        # 冻结 BERT 参数
        for param in self.bert.parameters():

            param.requires_grad = False

        self.bert.eval()

    def train(self, mode=True):

        super().train(mode)

        # 保持 BERT 的 eval 状态
        self.bert.eval()

        return self

    def forward(
        self,
        input_ids,
        attention_mask,
        token_type_ids
    ):

        # BERT 不参与梯度计算
        with torch.no_grad():

            outputs = self.bert(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )

        # 获取 CLS 向量
        features = outputs.last_hidden_state[:, 0]

        # 分类
        logits = self.fc(features)

        return logits


model = Model(pretrained_model).to(device)


# ============================================================
# 8. 评估函数
# ============================================================

def evaluate(model, dataloader, criterion):

    model.eval()

    total_loss = 0

    all_predictions = []
    all_labels = []

    with torch.no_grad():

        for batch in dataloader:

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            token_type_ids = batch['token_type_ids'].to(device)
            labels = batch['labels'].to(device)

            logits = model(
                input_ids,
                attention_mask,
                token_type_ids
            )

            loss = criterion(
                logits,
                labels
            )

            predictions = logits.argmax(dim=1)

            total_loss += loss.item() * labels.size(0)

            all_predictions.extend(
                predictions.cpu().tolist()
            )

            all_labels.extend(
                labels.cpu().tolist()
            )

    avg_loss = total_loss / len(all_labels)

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    precision = precision_score(
        all_labels,
        all_predictions,
        average='macro',
        zero_division=0
    )

    recall = recall_score(
        all_labels,
        all_predictions,
        average='macro',
        zero_division=0
    )

    f1 = f1_score(
        all_labels,
        all_predictions,
        average='macro',
        zero_division=0
    )

    return {
        'loss': avg_loss,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'predictions': all_predictions,
        'labels': all_labels
    }


# ============================================================
# 9. 训练函数
# ============================================================

def train(epochs=EPOCHS):

    optimizer = torch.optim.AdamW(
        filter(
            lambda p: p.requires_grad,
            model.parameters()
        ),
        lr=LEARNING_RATE
    )

    criterion = nn.CrossEntropyLoss()

    total_steps = len(train_loader) * epochs

    scheduler = get_scheduler(
        name='linear',
        num_warmup_steps=0,
        num_training_steps=total_steps,
        optimizer=optimizer
    )

    best_f1 = -1

    patience_counter = 0

    history = {
        'train_loss': [],
        'train_accuracy': [],
        'val_loss': [],
        'val_accuracy': [],
        'val_f1': []
    }

    # ========================================================
    # Epoch 循环
    # ========================================================

    for epoch in range(epochs):

        model.train()

        total_loss = 0
        total_correct = 0
        total_samples = 0

        # ====================================================
        # 训练阶段
        # ====================================================

        for step, batch in enumerate(train_loader):

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            token_type_ids = batch['token_type_ids'].to(device)
            labels = batch['labels'].to(device)

            # 前向传播
            logits = model(
                input_ids,
                attention_mask,
                token_type_ids
            )

            # 计算损失
            loss = criterion(
                logits,
                labels
            )

            # 反向传播
            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            scheduler.step()

            # 统计训练结果
            predictions = logits.argmax(dim=1)

            total_loss += loss.item() * labels.size(0)

            total_correct += (
                predictions == labels
            ).sum().item()

            total_samples += labels.size(0)

        train_loss = total_loss / total_samples

        train_accuracy = (
            total_correct / total_samples
        )

        # ====================================================
        # 验证阶段
        # ====================================================

        val_metrics = evaluate(
            model,
            val_loader,
            criterion
        )

        history['train_loss'].append(train_loss)
        history['train_accuracy'].append(train_accuracy)
        history['val_loss'].append(val_metrics['loss'])
        history['val_accuracy'].append(val_metrics['accuracy'])
        history['val_f1'].append(val_metrics['f1'])

        print('\n' + '=' * 55)

        print(
            f'Epoch: {epoch + 1}/{epochs}'
        )

        print(
            f'Train Loss: {train_loss:.4f}'
        )

        print(
            f'Train Accuracy: {train_accuracy:.4f}'
        )

        print(
            f'Val Loss: {val_metrics["loss"]:.4f}'
        )

        print(
            f'Val Accuracy: {val_metrics["accuracy"]:.4f}'
        )

        print(
            f'Val Precision: {val_metrics["precision"]:.4f}'
        )

        print(
            f'Val Recall: {val_metrics["recall"]:.4f}'
        )

        print(
            f'Val F1: {val_metrics["f1"]:.4f}'
        )

        print(
            f'Learning Rate: {scheduler.get_last_lr()[0]:.8f}'
        )

        # ====================================================
        # 保存最佳模型
        # ====================================================

        if val_metrics['f1'] > best_f1:

            best_f1 = val_metrics['f1']

            patience_counter = 0

            torch.save(
                {
                    'model_state_dict': model.state_dict(),
                    'best_f1': best_f1,
                    'epoch': epoch + 1,
                    'model_name': MODEL_NAME,
                    'max_length': MAX_LENGTH
                },
                MODEL_PATH
            )

            print('已保存最佳模型！')

        else:

            patience_counter += 1

            print(
                f'验证集 F1 未提升，'
                f'早停计数: {patience_counter}/{PATIENCE}'
            )

        # ====================================================
        # Early Stopping
        # ====================================================

        if patience_counter >= PATIENCE:

            print('触发 Early Stopping，停止训练！')

            break

    print('\n训练结束！')

    print(f'最佳验证集 F1: {best_f1:.4f}')

    # 加载最佳模型
    checkpoint = torch.load(
        MODEL_PATH,
        map_location=device,
        weights_only=True
    )

    model.load_state_dict(
        checkpoint['model_state_dict']
    )

    return history


# ============================================================
# 10. 训练过程可视化
# ============================================================

def plot_history(history):

    epochs = range(
        1,
        len(history['train_loss']) + 1
    )

    plt.figure(figsize=(12, 5))

    # Loss 曲线
    plt.subplot(1, 2, 1)

    plt.plot(
        epochs,
        history['train_loss'],
        label='Train Loss'
    )

    plt.plot(
        epochs,
        history['val_loss'],
        label='Validation Loss'
    )

    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()

    # Accuracy 曲线
    plt.subplot(1, 2, 2)

    plt.plot(
        epochs,
        history['train_accuracy'],
        label='Train Accuracy'
    )

    plt.plot(
        epochs,
        history['val_accuracy'],
        label='Validation Accuracy'
    )

    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.legend()

    plt.tight_layout()

    plt.show()


# ============================================================
# 11. 测试函数
# ============================================================

def test(model):

    checkpoint = torch.load(
        MODEL_PATH,
        map_location=device,
        weights_only=True
    )

    model.load_state_dict(
        checkpoint['model_state_dict']
    )

    criterion = nn.CrossEntropyLoss()

    metrics = evaluate(
        model,
        test_loader,
        criterion
    )

    print('\n========== 最终测试结果 ==========')

    print(f'Loss:      {metrics["loss"]:.4f}')
    print(f'Accuracy:  {metrics["accuracy"]:.4f}')
    print(f'Precision: {metrics["precision"]:.4f}')
    print(f'Recall:    {metrics["recall"]:.4f}')
    print(f'F1:        {metrics["f1"]:.4f}')

    # 分类报告
    print('\n========== 分类报告 ==========')

    print(
        classification_report(
            metrics['labels'],
            metrics['predictions'],
            labels=[0, 1],
            target_names=LABEL_NAMES,
            zero_division=0
        )
    )

    # 混淆矩阵
    cm = confusion_matrix(
        metrics['labels'],
        metrics['predictions'],
        labels=[0, 1]
    )

    print('\n========== 混淆矩阵 ==========')

    print(cm)

    plt.figure(figsize=(7, 5))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=LABEL_NAMES,
        yticklabels=LABEL_NAMES
    )

    plt.xlabel('Predicted')
    plt.ylabel('Actual')

    plt.title('Confusion Matrix')

    plt.show()

    return metrics


# ============================================================
# 12. 单条文本预测
# ============================================================

def predict(text):

    model.eval()

    data = tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
        return_tensors='pt'
    )

    input_ids = data['input_ids'].to(device)

    attention_mask = data['attention_mask'].to(device)

    token_type_ids = data['token_type_ids'].to(device)

    with torch.no_grad():

        logits = model(
            input_ids,
            attention_mask,
            token_type_ids
        )

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        prediction = probabilities.argmax(
            dim=1
        ).item()

    result = {
        'text': text,
        'label': prediction,
        'sentiment': LABEL_NAMES[prediction],
        'confidence': probabilities[0][prediction].item(),
        'probabilities': {
            LABEL_NAMES[i]: probabilities[0][i].item()
            for i in range(len(LABEL_NAMES))
        }
    }

    return result


cpu


In [ ]:
# 训练
history = train(epochs=EPOCHS)

# 绘制训练曲线
plot_history(history)

# 最终测试
test_metrics = test(model)

# 单条文本预测
print('\n========== 单条文本预测 ==========')

examples = [
    '这个产品质量非常好，我很满意。',
    '服务态度太差了，再也不会购买。',
    '东西还可以，没有特别惊喜。',
    '非常糟糕，完全不值得购买。'
]

for text in examples:

    result = predict(text)

    print('\n文本:', result['text'])

    print('情感:', result['sentiment'])

    print(
        '置信度:',
        f'{result["confidence"]:.2%}'
    )

    print(
        '概率:',
        result['probabilities']
    )

In [28]:

import torch
import sys

print('Python 路径:', sys.executable)
print('PyTorch 版本:', torch.__version__)

print('CUDA 是否可用:', torch.cuda.is_available())
print('PyTorch CUDA 版本:', torch.version.cuda)

print('GPU 数量:', torch.cuda.device_count())

if torch.cuda.is_available():
    print('GPU 名称:', torch.cuda.get_device_name(0))
    print('当前 GPU:', torch.cuda.current_device())
else:
    print('当前只能使用 CPU')


Python 路径: E:\py_workspace\02_huggingface\.venv\Scripts\python.exe
PyTorch 版本: 2.15.0+cpu
CUDA 是否可用: False
PyTorch CUDA 版本: None
GPU 数量: 0
当前只能使用 CPU
